# 03 · Foundry IQ e agenti

Notebook che crea gli oggetti Foundry IQ e gli agenti del progetto Foundry dopo i notebook 01 e 02. La configurazione arriva dagli output ARM; la chiave MCP è letta dall'App Service e scritta solo nelle definizioni che la richiedono (knowledge source MCP e connessione del progetto).

| Oggetto | Tipo | Use case |
| --- | --- | --- |
| `machinespecs-ks` | azureBlob | Manuali (multi-fonte e RAG classico) |
| `ricambi-magazzino-ks` | mcpServer | Magazzino ricambi |
| `policies-sharepoint-ks` | remoteSharePoint (opzionale) | Policy con permessi utente |
| `policies-sharepoint-indexed-ks` | indexedSharePoint (opzionale) | Policy indicizzate con ACL |
| `bing-filtered-ks` | web | Cataloghi di distributori consentiti |
| `wpc-foundryiq-kb` | knowledge base | Demo multi-fonte e agente Foundry IQ |
| `acl-docs-ks` + `wpc-acl-kb` | ADLS Gen2 con ACL | Demo ACL |
| `wpc-foundryiq-kb-mcp`, `mcp-ricambi-key` | connessioni progetto | Tool degli agenti |
| Agenti `agents/*.yaml` | prompt agent | Demo agenti |

In [ ]:
# Configurazione: usa lo stesso ENVIRONMENT_NAME degli altri notebook.
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: serve infra/main.bicep.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from notebook_env import load_azd_env

LOAD_AZD_ENV = False               # True = precompila con l'ambiente azd
AZD_ENVIRONMENT_NAME = ""           # vuoto = ambiente selezionato; oppure un nome esplicito
azd_values = load_azd_env(AZD_ENVIRONMENT_NAME, root=ROOT, override=True) if LOAD_AZD_ENV else {}
if LOAD_AZD_ENV:
    missing = [k for k in ("AZURE_ENV_NAME", "AZURE_SUBSCRIPTION_ID", "AZURE_TENANT_ID") if not azd_values.get(k)]
    if missing:
        raise RuntimeError(f"Valori mancanti nell'ambiente azd: {', '.join(missing)}.")

SUBSCRIPTION_ID = azd_values.get("AZURE_SUBSCRIPTION_ID", "")
ENVIRONMENT_NAME = azd_values.get("AZURE_ENV_NAME") or "wpc-demo"
DEPLOYMENT_NAME = azd_values.get("AZURE_DEPLOYMENT_NAME") or f"wpc-foundryiq-{ENVIRONMENT_NAME}"
SEARCH_API_VERSION = "2026-11-01-preview"

KB_NAME = azd_values.get("FOUNDRYIQ_DEMO_KB_NAME") or "wpc-foundryiq-kb"
BLOB_KS = "machinespecs-ks"
MCP_KS = "ricambi-magazzino-ks"
SHAREPOINT_KS = "policies-sharepoint-ks"
INDEXED_SHAREPOINT_KS = "policies-sharepoint-indexed-ks"
WEB_KS = "bing-filtered-ks"
ACL_KS = "acl-docs-ks"
ACL_KB_NAME = azd_values.get("FOUNDRYIQ_ACL_KB_NAME") or "wpc-acl-kb"
KB_CONNECTION = "wpc-foundryiq-kb-mcp"
MCP_CONNECTION = "mcp-ricambi-key"
BING_ALLOWED_DOMAINS = ["grainger.com", "mcmaster.com", "rs-online.com", "digikey.com", "mouser.com", "misumi-ec.com", "automationdirect.com"]

# SharePoint: login delegato Microsoft Graph (device code) nello stesso tenant di Azure.
RUN_SHAREPOINT = True
SHAREPOINT_SITE_URL = ""                # vuoto = https://<tenant>.sharepoint.com/sites/<alias>
SHAREPOINT_SITE_ALIAS = "wpc-foundry-iq-demo"
SHAREPOINT_FOLDER = "policies"
SHAREPOINT_POLICIES_URL = azd_values.get("SHAREPOINT_POLICIES_URL", "")  # URL cartella già pronta, salta il caricamento
GRAPH_CLIENT_ID = "14d82eec-204b-4c2f-b7e8-296a70dab67e"  # Microsoft Graph Command Line Tools (client pubblico)
# SharePoint indexed: crea l'app di ingestion secretless (serve un amministratore Entra per il consenso).
# Una SHAREPOINT_INDEXED_CONNECTION_STRING già presente nell'ambiente del kernel ha la precedenza.
RUN_SHAREPOINT_INDEXED = True
SHAREPOINT_INDEXED_QUERY = ""           # vuoto = includeFolder=<SHAREPOINT_POLICIES_URL>

RUN_ACL_KB = True
RUN_AGENTS = True

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: servono infra/, data/, agents/ e le app.")
AZ = shutil.which("az")
if AZ is None:
    raise RuntimeError("Azure CLI non trovata nel PATH: installala, esegui 'az login' e riavvia il kernel.")


def az(*args, output=True, check=True):
    """Esegue Azure CLI. Con output=True restituisce il JSON decodificato; con check=False None in caso di errore."""
    command = [AZ, *args, "--only-show-errors"] + (["--output", "json"] if output else [])
    proc = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        if check:
            raise RuntimeError(f"az {' '.join(args[:3])} fallito: {proc.stderr.strip()[-2000:]}")
        return None
    text = proc.stdout.strip()
    return (json.loads(text) if text else None) if output else text


def load_outputs(deployment_name):
    """Legge gli output del deployment ARM di subscription creato da 01-infrastruttura.ipynb."""
    deployment = az("deployment", "sub", "show", "--name", deployment_name, check=False)
    if not deployment:
        raise RuntimeError(f"Deployment '{deployment_name}' non trovato nella subscription attiva: esegui 01-infrastruttura.ipynb.")
    state = deployment["properties"].get("provisioningState")
    if state != "Succeeded":
        raise RuntimeError(f"Deployment '{deployment_name}' in stato {state}: correggi e riesegui 01-infrastruttura.ipynb.")
    return {name.upper(): item["value"] for name, item in (deployment["properties"].get("outputs") or {}).items()}


if SUBSCRIPTION_ID:
    az("account", "set", "--subscription", SUBSCRIPTION_ID, output=False)
account = az("account", "show", check=False)
if not account:
    raise RuntimeError("Azure CLI non autenticata: esegui 'az login' (eventualmente con --tenant).")
if azd_values.get("AZURE_TENANT_ID") and azd_values["AZURE_TENANT_ID"].casefold() != account["tenantId"].casefold():
    raise RuntimeError("Il tenant dell'ambiente azd non corrisponde alla subscription selezionata in Azure CLI.")
SUBSCRIPTION_ID, TENANT_ID = account["id"], account["tenantId"]
out = load_outputs(DEPLOYMENT_NAME)
if out["AZURE_TENANT_ID"] != TENANT_ID:
    raise RuntimeError(f"Il deployment usa il tenant {out['AZURE_TENANT_ID']}, Azure CLI il tenant {TENANT_ID}.")
RESOURCE_GROUP = out["AZURE_RESOURCE_GROUP"]
print(f"Subscription: {account['name']} ({SUBSCRIPTION_ID})")
print(f"Deployment:   {DEPLOYMENT_NAME} -> resource group {RESOURCE_GROUP}")

TRANSIENT = re.compile(r"\b(401|403|404|408|409|429|500|502|503|504)\b|Unauthorized|Forbidden|AuthorizationPermissionMismatch|"
                       r"AuthorizationFailed|TooManyRequests|ServiceUnavailable|timed? ?out|temporarily|Error executing tool", re.I)


def retry(label, action, attempts=8, delay=20):
    """Ripete gli errori transitori: subito dopo il deploy i ruoli RBAC impiegano minuti a propagarsi."""
    for attempt in range(1, attempts + 1):
        try:
            return action()
        except Exception as exc:  # noqa: BLE001 - riprova solo gli errori transitori riconosciuti
            if attempt == attempts or not TRANSIENT.search(str(exc)):
                raise
            print(f"  {label}: errore transitorio ({attempt}/{attempts}), riprovo tra {delay} s...")
            time.sleep(delay)

import requests
from azure.identity import AzureCliCredential

credential = AzureCliCredential(tenant_id=TENANT_ID)
SEARCH = out["FOUNDRYIQ_DEMO_SEARCH_ENDPOINT"].rstrip("/")
created = []


def search_request(method, path, body=None):
    token = credential.get_token("https://search.azure.com/.default").token
    response = requests.request(method, f"{SEARCH}/{path}?api-version={SEARCH_API_VERSION}", json=body, timeout=120,
                                headers={"Authorization": f"Bearer {token}"})
    if method == "GET" and response.status_code == 404:
        return None
    if not response.ok:
        raise RuntimeError(f"{method} {path}: HTTP {response.status_code} {response.text[:800]}")
    return response.json() if response.content else {}


def search_put(path, body, kind):
    retry(path, lambda: search_request("PUT", path, body))
    created.append({"oggetto": path.split("/", 1)[1], "tipo": kind, "dove": "Azure AI Search"})
    print("  ok:", path)


def arm_put(url, body, kind):
    def call():
        token = credential.get_token("https://management.azure.com/.default").token
        response = requests.put(url, json=body, timeout=120, headers={"Authorization": f"Bearer {token}"})
        if not response.ok:
            raise RuntimeError(f"PUT {url.split('?')[0]}: HTTP {response.status_code} {response.text[:800]}")
    retry(body["name"], call)
    created.append({"oggetto": body["name"], "tipo": kind, "dove": "progetto Foundry"})
    print("  ok:", body["name"])


settings = az("webapp", "config", "appsettings", "list", "--resource-group", RESOURCE_GROUP, "--name", out["MCP_APP_NAME"])
mcp_key = next(s["value"] for s in settings if s["name"] == "MCP_API_KEY")
del settings
print("Chiave MCP letta dall'App Service (in memoria, non stampata).")

## 1 · Policy su SharePoint (opzionale)

Il knowledge source *remote SharePoint* interroga SharePoint in tempo reale con il token dell'utente finale, quindi rispetta i permessi del sito. La cella trova o crea il communication site, carica i PDF di `data/sharepoint/policies/EN` e `IT` nelle rispettive sottocartelle (solo i file cambiati) e usa la cartella principale come filtro. Il login Graph usa il **device code**: segui le istruzioni stampate.

In [ ]:
from urllib.parse import quote, unquote, urlsplit

if not SHAREPOINT_POLICIES_URL and RUN_SHAREPOINT:
    from azure.identity import DeviceCodeCredential

    policy_files = {}
    for language in ("EN", "IT"):
        source = ROOT / "data" / "sharepoint" / "policies" / language
        policy_files[language] = sorted(source.glob("*.pdf"))
        if not policy_files[language]:
            raise RuntimeError(f"Nessun PDF da caricare in {source}")

    graph_credential = DeviceCodeCredential(client_id=GRAPH_CLIENT_ID, tenant_id=TENANT_ID)
    graph_scopes = ["https://graph.microsoft.com/Sites.ReadWrite.All"]

    def graph(method, path, **kwargs):
        token = graph_credential.get_token(*graph_scopes).token
        headers = {"Authorization": f"Bearer {token}", **kwargs.pop("headers", {})}
        response = requests.request(method, f"https://graph.microsoft.com/{path}", headers=headers, timeout=120, **kwargs)
        if method == "GET" and response.status_code == 404:
            return None
        if not response.ok:
            raise RuntimeError(f"Graph {method} {path.split('?')[0]}: HTTP {response.status_code} {response.text[:500]}")
        return response.json() if response.content else {}

    site_url = SHAREPOINT_SITE_URL.rstrip("/")
    if not site_url:
        host = graph("GET", "v1.0/sites/root?$select=siteCollection")["siteCollection"]["hostname"]
        site_url = f"https://{host}/sites/{SHAREPOINT_SITE_ALIAS}"
    parts = urlsplit(site_url)
    lookup = f"v1.0/sites/{parts.hostname}:{parts.path}?$select=id,webUrl,displayName"
    site = graph("GET", lookup)
    if site is None:
        print(f"Creo il communication site {site_url}...")
        graph_scopes = ["https://graph.microsoft.com/Sites.ReadWrite.All", "https://graph.microsoft.com/Sites.Create.All"]
        graph("POST", "beta/sites", json={"name": "WPC Foundry IQ demo", "webUrl": site_url, "locale": "en-US",
                                          "description": "Documenti per la demo Foundry IQ (remote SharePoint knowledge source).",
                                          "shareByEmailEnabled": False, "template": "sitepagepublishing"})
        for _ in range(30):
            time.sleep(10)
            site = graph("GET", lookup)
            if site:
                break
        if site is None:
            raise RuntimeError("Sito non ancora disponibile: riesegui la cella tra qualche minuto.")
    drive = graph("GET", f"v1.0/sites/{site['id']}/drive?$select=id,name")
    drive_path = f"v1.0/drives/{drive['id']}"
    folder = quote(SHAREPOINT_FOLDER, safe="")
    folder_item = graph("GET", f"{drive_path}/root:/{folder}?$select=id,folder,webUrl")
    if folder_item is None:
        folder_item = graph("POST", f"{drive_path}/root/children", json={"name": SHAREPOINT_FOLDER, "folder": {},
                            "@microsoft.graph.conflictBehavior": "fail"})
    if "folder" not in folder_item:
        raise RuntimeError(f"Il percorso SharePoint {SHAREPOINT_FOLDER!r} non è una cartella")
    uploaded = 0
    for language, files in policy_files.items():
        language_item = graph("GET", f"{drive_path}/items/{folder_item['id']}:/{language}?$select=id,folder")
        if language_item is None:
            language_item = graph("POST", f"{drive_path}/items/{folder_item['id']}/children",
                                  json={"name": language, "folder": {}, "@microsoft.graph.conflictBehavior": "fail"})
        if "folder" not in language_item:
            raise RuntimeError(f"Il percorso SharePoint {SHAREPOINT_FOLDER}/{language} non è una cartella")
        current = graph("GET", f"{drive_path}/items/{language_item['id']}/children?$select=name,size&$top=999")
        sizes = {item["name"]: item["size"] for item in current["value"]}
        for path in files:
            if sizes.get(path.name) == path.stat().st_size:
                continue
            graph("PUT", f"{drive_path}/items/{language_item['id']}:/{quote(path.name, safe='')}:/content",
                  data=path.read_bytes(), headers={"Content-Type": "application/octet-stream"})
            uploaded += 1
    SHAREPOINT_POLICIES_URL = unquote(folder_item["webUrl"])
    print(f"Sito {site['webUrl']}: {uploaded} file caricati.")
elif not SHAREPOINT_POLICIES_URL:
    existing = search_request("GET", f"knowledgesources/{SHAREPOINT_KS}") or {}
    match = re.search(r'Path:"([^"]+)"', (existing.get("remoteSharePointParameters") or {}).get("filterExpression") or "")
    SHAREPOINT_POLICIES_URL = match.group(1) if match else ""
print("Cartella policy:", SHAREPOINT_POLICIES_URL or "non configurata: la KB userà manuali, ricambi e web")

## 1b · App di ingestion per SharePoint indexed (opzionale)

Il knowledge source *indexed SharePoint* importa contenuti e ACL in Search con un'app Entra con **application permissions** (`Files.Read.All`, `Sites.FullControl.All`), separata dal login Graph usato per l'upload. La cella crea o aggiorna l'app `wpc-foundryiq-sp-ingestion-<ambiente>`, concede il consenso amministrativo (serve Global Administrator o Privileged Role Administrator) e aggiunge una credenziale federata che si fida dell'identità gestita system-assigned di Search: nessun segreto. La connection string resta solo in memoria nel kernel. Imposta `RUN_SHAREPOINT_INDEXED = False` per saltare il passo; dettagli in [SharePoint indexed](../../infra/README.md#sharepoint-indexed).

In [ ]:
from urllib.parse import urlsplit

GRAPH_APP_ID = "00000003-0000-0000-c000-000000000000"
GRAPH_ROLES = {"Files.Read.All": "01d4889c-1287-42c6-ac1f-5d1e02578ef6",
               "Sites.FullControl.All": "a82116e5-55eb-4c41-a434-62fe8a61c773"}

if os.environ.get("SHAREPOINT_INDEXED_CONNECTION_STRING"):
    print("SharePoint indexed: uso la connessione già presente nell'ambiente del kernel.")
elif not RUN_SHAREPOINT_INDEXED or not SHAREPOINT_POLICIES_URL:
    print("SharePoint indexed saltato: RUN_SHAREPOINT_INDEXED=False o cartella policy non configurata.")
else:
    parts = urlsplit(SHAREPOINT_POLICIES_URL)
    segments = parts.path.strip("/").split("/")
    sp_site_url = SHAREPOINT_SITE_URL.rstrip("/") or f"{parts.scheme}://{parts.netloc}" + (
        "/" + "/".join(segments[:2]) if segments[0] in ("sites", "teams") else "")

    search_principal_id = out.get("SEARCH_PRINCIPAL_ID") or az(
        "search", "service", "show", "--resource-group", RESOURCE_GROUP, "--name", out["SEARCH_SERVICE_NAME"],
        "--query", "identity.principalId")
    if not search_principal_id:
        raise RuntimeError("Il servizio Search non ha un'identità gestita system-assigned.")
    search_client_id = az("ad", "sp", "show", "--id", search_principal_id, "--query", "appId")

    ingestion_name = f"wpc-foundryiq-sp-ingestion-{ENVIRONMENT_NAME}"
    ingestion_app_id = az("ad", "app", "list", "--display-name", ingestion_name, "--query", "[0].appId")
    if not ingestion_app_id:
        ingestion_app_id = az("ad", "app", "create", "--display-name", ingestion_name,
                              "--sign-in-audience", "AzureADMyOrg", "--query", "appId")
        print(f"App di ingestion creata: {ingestion_name} ({ingestion_app_id})")
    az("ad", "app", "update", "--id", ingestion_app_id, "--required-resource-accesses", json.dumps([{
        "resourceAppId": GRAPH_APP_ID, "resourceAccess": [{"id": r, "type": "Role"} for r in GRAPH_ROLES.values()]}]),
       output=False)
    ingestion_sp_id = az("ad", "sp", "show", "--id", ingestion_app_id, "--query", "id", check=False) or az(
        "ad", "sp", "create", "--id", ingestion_app_id, "--query", "id")

    # Consenso amministrativo: assegna i ruoli applicativi di Microsoft Graph al service principal.
    graph_sp_id = az("ad", "sp", "show", "--id", GRAPH_APP_ID, "--query", "id")
    assignments_uri = f"https://graph.microsoft.com/v1.0/servicePrincipals/{ingestion_sp_id}/appRoleAssignments"
    granted = None
    for _ in range(6):  # un service principal appena creato può non essere ancora visibile a Graph
        granted = az("rest", "--method", "get", "--uri", assignments_uri,
                     "--query", f"value[?resourceId=='{graph_sp_id}'].appRoleId", check=False)
        if granted is not None:
            break
        time.sleep(10)
    if granted is None:
        raise RuntimeError("Impossibile leggere le assegnazioni di ruolo dell'app di ingestion.")
    for role, role_id in GRAPH_ROLES.items():
        if role_id in granted:
            continue
        body = json.dumps({"principalId": ingestion_sp_id, "resourceId": graph_sp_id, "appRoleId": role_id})
        if az("rest", "--method", "post", "--uri", assignments_uri, "--body", body,
              "--headers", "Content-Type=application/json", output=False, check=False) is None:
            raise RuntimeError(f"Consenso per {role} non concesso: servono Global Administrator o Privileged Role "
                               f"Administrator. Concedilo dal portale Entra (App registrations > {ingestion_name} > "
                               "API permissions > Grant admin consent) e riesegui la cella.")
        print(f"  {role} concesso.")

    fic_name = "search-managed-identity"
    fic_subject = az("ad", "app", "federated-credential", "list", "--id", ingestion_app_id,
                     "--query", f"[?name=='{fic_name}'].subject | [0]")
    if fic_subject and fic_subject != search_principal_id:
        # Servizio Search ricreato: la credenziale deve fidarsi della nuova identità.
        az("ad", "app", "federated-credential", "delete", "--id", ingestion_app_id,
           "--federated-credential-id", fic_name, output=False)
        fic_subject = None
    if not fic_subject:
        az("ad", "app", "federated-credential", "create", "--id", ingestion_app_id, "--parameters", json.dumps({
            "name": fic_name, "issuer": f"https://login.microsoftonline.com/{TENANT_ID}/v2.0",
            "subject": search_principal_id, "audiences": ["api://AzureADTokenExchange"]}), output=False)

    os.environ["SHAREPOINT_INDEXED_CONNECTION_STRING"] = (
        f"SharePointOnlineEndpoint={sp_site_url};ApplicationId={ingestion_app_id};"
        f"FederatedCredentialApplicationId={search_client_id};TenantId={TENANT_ID}")
    print(f"App di ingestion SharePoint pronta: {ingestion_name} (nessun segreto).")

## 2 · Knowledge source

Ogni knowledge source descrive *una* fonte e come interrogarla. La KB multi-fonte decide poi quali usare per ogni domanda.

SharePoint indexed usa la connessione preparata nella sezione precedente (o `SHAREPOINT_INDEXED_CONNECTION_STRING` già presente nell'ambiente del kernel). La cartella e le sottocartelle vengono indicizzate con ACL `userIds`/`groupIds`, senza sostituire la sorgente remota. Verifica il completamento dell'indexer in Search: contenuti e ACL riflettono l'ultima ingestion (default ogni 24 ore). Il client deve inoltrare il token utente per accedere ai documenti protetti; i gruppi nativi SharePoint richiedono configurazione aggiuntiva del servizio, descritta in [SharePoint indexed](../../infra/README.md#sharepoint-indexed).

In [ ]:
indexed_sharepoint_connection = os.environ.get("SHAREPOINT_INDEXED_CONNECTION_STRING", "")
indexed_sharepoint_query = SHAREPOINT_INDEXED_QUERY or os.environ.get("SHAREPOINT_INDEXED_QUERY", "")
if indexed_sharepoint_connection and not indexed_sharepoint_query:
    if not SHAREPOINT_POLICIES_URL:
        raise ValueError("Configura SHAREPOINT_POLICIES_URL oppure SHAREPOINT_INDEXED_QUERY prima di creare le sorgenti.")
    indexed_sharepoint_query = f"includeFolder={SHAREPOINT_POLICIES_URL}"

openai_endpoint = out["FOUNDRYIQ_DEMO_OPENAI_ENDPOINT"]
chat_model = {"kind": "azureOpenAI", "azureOpenAIParameters": {
    "resourceUri": openai_endpoint, "deploymentId": out["FOUNDRYIQ_DEMO_CHAT_DEPLOYMENT"], "modelName": out["FOUNDRYIQ_DEMO_CHAT_MODEL"]}}
embedding_model = {"kind": "azureOpenAI", "azureOpenAIParameters": {
    "resourceUri": openai_endpoint, "deploymentId": out["FOUNDRYIQ_DEMO_EMBEDDING_DEPLOYMENT"], "modelName": "text-embedding-3-large"}}

search_put(f"knowledgesources/{BLOB_KS}", {
    "name": BLOB_KS, "kind": "azureBlob",
    "description": "Manuali tecnici delle macchine (specifiche, componenti, manutenzione, troubleshooting).",
    "azureBlobParameters": {
        "connectionString": f"ResourceId={out['FOUNDRYIQ_DEMO_STORAGE_RESOURCE_ID']};",
        "containerName": out["FOUNDRYIQ_DEMO_BLOB_CONTAINER"], "isADLSGen2": False,
        "ingestionParameters": {"contentExtractionMode": "minimal", "embeddingModel": embedding_model, "chatCompletionModel": chat_model}}},
    "azureBlob")
# Gli agenti RAG classico interrogano direttamente l'indice generato dal knowledge source.
blob_ks = search_request("GET", f"knowledgesources/{BLOB_KS}") or {}
blob_index = ((blob_ks.get("azureBlobParameters") or {}).get("createdResources") or {}).get("index") or f"{BLOB_KS}-index"
print("  indice manuali per il RAG classico:", blob_index)

# Foundry IQ accetta al massimo 5 tool per sorgente MCP: list_machine_parts è coperto da search_parts.
mcp_tools = ["search_parts", "get_part_details", "check_stock", "find_alternative_parts", "low_stock_report"]
search_put(f"knowledgesources/{MCP_KS}", {
    "name": MCP_KS, "kind": "mcpServer", "resultsProcessing": "rerank",
    "description": "Catalogo ricambi e giacenze di magazzino: ricerca ricambi, disponibilità, alternative, ordini aperti, ricambi sotto scorta.",
    "mcpServerParameters": {
        "serverURL": out["MCP_SERVER_URL"],
        "authentication": {"kind": "storedHeaders", "storedHeadersParameters": {"headers": {"x-api-key": mcp_key}}},
        "tools": [{"name": t, "resultsProcessing": "none", "maxOutputTokens": 2000,
                   "outputParsing": {"kind": "json", "jsonParameters": {"documentsPath": "$.results[*]", "includeContext": False}}}
                  for t in mcp_tools]}},
    "mcpServer")

domain_pattern = re.compile(r"^(?=.{1,253}$)(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,63}$")
domains = [d.strip().lower() for d in BING_ALLOWED_DOMAINS if d.strip()]
if not domains or not all(domain_pattern.match(d) for d in domains):
    raise ValueError("BING_ALLOWED_DOMAINS deve contenere nomi host validi, senza schema o percorso.")
search_put(f"knowledgesources/{WEB_KS}", {
    "name": WEB_KS, "kind": "web",
    "description": "Ricerca web Bing limitata ai cataloghi di distributori tecnici consentiti per informazioni aggiornate sui ricambi.",
    "webParameters": {"domains": {"allowedDomains": [{"address": d, "includeSubpages": True} for d in domains]}}},
    "web")

if SHAREPOINT_POLICIES_URL:
    search_put(f"knowledgesources/{SHAREPOINT_KS}", {
        "name": SHAREPOINT_KS, "kind": "remoteSharePoint",
        "description": "Policy aziendali per i tecnici (sicurezza, riparare o sostituire, emergenze, comunicazione, conformità) su SharePoint.",
        "remoteSharePointParameters": {"filterExpression": f'Path:"{SHAREPOINT_POLICIES_URL}"',
                                       "resourceMetadata": ["Author", "Title"], "containerTypeId": None}},
        "remoteSharePoint")

if indexed_sharepoint_connection:
    search_put(f"knowledgesources/{INDEXED_SHAREPOINT_KS}", {
        "name": INDEXED_SHAREPOINT_KS, "kind": "indexedSharePoint",
        "description": "Policy aziendali SharePoint indicizzate con permessi documentali importati.",
        "indexedSharePointParameters": {
            "connectionString": indexed_sharepoint_connection, "containerName": "useQuery",
            "query": indexed_sharepoint_query,
            "ingestionParameters": {"contentExtractionMode": "minimal", "embeddingModel": embedding_model,
                                    "ingestionPermissionOptions": ["userIds", "groupIds"]}}},
        "indexedSharePoint")
else:
    print("SharePoint indexed saltato: SHAREPOINT_INDEXED_CONNECTION_STRING non configurata.")

## 3 · Knowledge base multi-fonte

Reasoning effort `low` (il valore `minimal` non è supportato con sorgenti MCP), output estrattivo e istruzioni che indicano quando usare ogni fonte.

In [ ]:
# Nella KB entra solo la sorgente SharePoint indicizzata (con ACL); quella remota non viene mai aggiunta.
kb_sharepoint_ks = INDEXED_SHAREPOINT_KS if indexed_sharepoint_connection else ""
sources = [BLOB_KS, MCP_KS] + ([kb_sharepoint_ks] if kb_sharepoint_ks else []) + [WEB_KS]
instructions = [
    "Pianifica ricerche distinte per gli aspetti della domanda e usa le sorgenti pertinenti; combina più sorgenti quando la richiesta lo richiede.",
    f"Usa '{BLOB_KS}' per manuali, specifiche delle macchine, manutenzione e troubleshooting.",
    f"Usa '{MCP_KS}' per ricambi, giacenze, sottoscorta, ordini e alternative; scegli i tool pertinenti come search_parts, check_stock, low_stock_report e find_alternative_parts.",
]
if kb_sharepoint_ks:
    instructions.append(f"Usa '{kb_sharepoint_ks}' per policy aziendali, sicurezza e procedure, rispettando i permessi dell'utente.")
instructions += [
    f"Usa '{WEB_KS}' per cataloghi, fornitori e informazioni pubbliche aggiornate; non includere dati riservati nelle ricerche web.",
    "Mantieni esatti i nomi delle macchine e i codici dei ricambi. Non presumere disponibilità, date di consegna o autorizzazioni non presenti nelle fonti.",
]
kb_chat_model = {"kind": "azureOpenAI", "azureOpenAIParameters": {
    "resourceUri": openai_endpoint, "deploymentId": out.get("FOUNDRYIQ_DEMO_GPT52_DEPLOYMENT", "gpt-5.2"), "modelName": "gpt-5.2"}}
search_put(f"knowledgebases/{KB_NAME}", {
    "name": KB_NAME,
    "description": "Assistente tecnici WPC: manuali macchine, ricambi e giacenze, policy aziendali e ricerca web Bing filtrata.",
    "knowledgeSources": [{"name": s} for s in sources], "models": [kb_chat_model],
    "retrievalReasoningEffort": {"kind": "low"}, "retrievalInstructions": "\n".join(instructions),
    "outputMode": "extractiveData", "retrieveDefaults": {"maxRuntimeInSeconds": 60, "maxOutputDocuments": 10}},
    "knowledge base")

## 4 · Knowledge base con ACL

Il knowledge source ADLS Gen2 ingerisce, per ogni documento, gli ID di utenti e gruppi delle ACL di origine (`ingestionPermissionOptions`). Al momento della retrieve Foundry IQ filtra i risultati con il token dell'utente.

In [ ]:
if RUN_ACL_KB:
    search_put(f"knowledgesources/{ACL_KS}", {
        "name": ACL_KS, "kind": "azureBlob",
        "description": "Documenti interni con permessi per documento (ACL ADLS Gen2): procedure pubbliche, codici service riservati ai tecnici, accordi fornitori riservati alla direzione.",
        "azureBlobParameters": {
            "connectionString": f"ResourceId={out['FOUNDRYIQ_ACL_STORAGE_RESOURCE_ID']};",
            "containerName": out["FOUNDRYIQ_ACL_CONTAINER"], "isADLSGen2": True,
            "ingestionParameters": {"contentExtractionMode": "minimal", "ingestionPermissionOptions": ["userIds", "groupIds"],
                                    "embeddingModel": embedding_model}}},
        "azureBlob ADLS + ACL")
    search_put(f"knowledgebases/{ACL_KB_NAME}", {
        "name": ACL_KB_NAME, "description": "Demo ACL Foundry IQ: ogni utente vede solo i documenti consentiti dalle ACL di origine.",
        "knowledgeSources": [{"name": ACL_KS}], "models": [chat_model],
        "retrievalReasoningEffort": {"kind": "minimal"}, "outputMode": "extractiveData"},
        "knowledge base")
else:
    print("RUN_ACL_KB=False: demo ACL non configurata.")

## 5 · Connessioni del progetto Foundry

- `wpc-foundryiq-kb-mcp`: endpoint MCP della knowledge base con la **managed identity del progetto** (tool `knowledge_base_retrieve`);
- `mcp-ricambi-key`: server MCP ricambi con header `x-api-key`; la chiave resta nella connessione, mai nella definizione dell'agente.

In [ ]:
kb_mcp_endpoint = f"{SEARCH}/knowledgebases/{KB_NAME}/mcp?api-version={SEARCH_API_VERSION}"
connections = f"https://management.azure.com{out['FOUNDRY_PROJECT_ID']}/connections"
arm_put(f"{connections}/{KB_CONNECTION}?api-version=2025-10-01-preview", {
    "name": KB_CONNECTION, "type": "Microsoft.MachineLearningServices/workspaces/connections",
    "properties": {"authType": "ProjectManagedIdentity", "category": "RemoteTool", "target": kb_mcp_endpoint,
                   "isSharedToAll": True, "audience": "https://search.azure.com/", "metadata": {"ApiType": "Azure"}}},
    "connessione RemoteTool (identità progetto)")
arm_put(f"{connections}/{MCP_CONNECTION}?api-version=2025-10-01-preview", {
    "name": MCP_CONNECTION, "type": "Microsoft.MachineLearningServices/workspaces/connections",
    "properties": {"authType": "CustomKeys", "category": "RemoteTool", "target": out["MCP_SERVER_URL"], "isSharedToAll": True,
                   "credentials": {"keys": {"x-api-key": mcp_key}}, "metadata": {"type": "custom_MCP"}}},
    "connessione RemoteTool (chiave)")
mcp_key = None  # la chiave non serve più

## 6 · Agenti

Le definizioni sono in [agents/*.yaml](../../agents/). I segnaposto `${VAR}` e `${VAR=default}` sono risolti **solo** con i valori calcolati in questo notebook. Una nuova versione è creata solo se la definizione cambia (hash `definition_sha256` nei metadati).

In [ ]:
import hashlib

PLACEHOLDER = re.compile(r"\$\{([A-Za-z_][A-Za-z0-9_]*)(?:=([^}]*))?\}")


def substitute(value, values, missing):
    if isinstance(value, dict):
        return {k: substitute(v, values, missing) for k, v in value.items()}
    if isinstance(value, list):
        return [substitute(v, values, missing) for v in value]
    if not isinstance(value, str):
        return value

    def replace(match):
        resolved = values.get(match.group(1)) or match.group(2)
        if resolved is None:
            missing.add(match.group(1))
            return match.group(0)
        return resolved

    return PLACEHOLDER.sub(replace, value)


def split_list(value):
    items = value if isinstance(value, list) else re.split(r"[,;]", str(value or ""))
    return [str(i).strip() for i in items if str(i).strip()]


def build_tool(tool):
    """Converte un tool YAML nel payload della definizione dell'agente."""
    kind = tool.get("type")
    if kind == "azure_ai_search":
        index = {"project_connection_id": tool["connection"], "index_name": tool["index_name"],
                 "query_type": tool.get("query_type", "simple")}
        if tool.get("top_k") is not None:
            index["top_k"] = int(tool["top_k"])
        if tool.get("filter"):
            index["filter"] = tool["filter"]
        return {"type": "azure_ai_search", "azure_ai_search": {"indexes": [index]}}
    if kind == "web_search":
        payload = {"type": "web_search"}
        if split_list(tool.get("allowed_domains")):
            payload["filters"] = {"allowed_domains": split_list(tool.get("allowed_domains"))}
        if tool.get("search_context_size"):
            payload["search_context_size"] = tool["search_context_size"]
        return payload
    if kind == "knowledge_base":
        payload = {"type": "mcp", "server_label": tool.get("server_label", "knowledge-base"), "server_url": tool["server_url"],
                   "require_approval": "never", "allowed_tools": ["knowledge_base_retrieve"],
                   "project_connection_id": tool["connection"]}
    elif kind == "mcp":
        payload = {"type": "mcp", "server_label": tool["server_label"], "server_url": tool["server_url"],
                   "require_approval": tool.get("require_approval", "never")}
        if tool.get("allowed_tools"):
            payload["allowed_tools"] = list(tool["allowed_tools"])
        if tool.get("connection"):
            payload["project_connection_id"] = tool["connection"]
    else:
        raise ValueError(f"tipo di tool non supportato: {kind!r}")
    if tool.get("description"):
        payload["server_description"] = tool["description"]
    return payload


def build_definition(spec):
    definition = {"kind": "prompt", "model": spec["model"], "instructions": spec["instructions"].strip(),
                  "tools": [build_tool(t) for t in spec.get("tools") or []]}
    if spec.get("temperature") is not None:
        definition["temperature"] = spec["temperature"]
    return definition

In [ ]:
if RUN_AGENTS:
    import yaml
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import (AISearchIndexResource, AzureAISearchTool, AzureAISearchToolResource, MCPTool,
                                          PromptAgentDefinition, WebSearchTool, WebSearchToolFilters)
    from azure.core.exceptions import ResourceNotFoundError

    values = {
        "FOUNDRYIQ_DEMO_CHAT_DEPLOYMENT": out["FOUNDRYIQ_DEMO_CHAT_DEPLOYMENT"],
        "FOUNDRYIQ_DEMO_KB_MCP_ENDPOINT": kb_mcp_endpoint,
        "FOUNDRYIQ_DEMO_KB_CONNECTION": KB_CONNECTION,
        "FOUNDRYIQ_MCP_CONNECTION": MCP_CONNECTION,
        "MCP_SERVER_URL": out["MCP_SERVER_URL"],
        "FOUNDRYIQ_SEARCH_CONNECTION_ID": out["FOUNDRYIQ_SEARCH_CONNECTION_ID"],
        "FOUNDRYIQ_DEMO_BLOB_INDEX": blob_index,
        "FOUNDRYIQ_BING_ALLOWED_DOMAINS": ",".join(domains),
    }

    def sdk_tool(tool):
        if tool["type"] == "azure_ai_search":
            indexes = [AISearchIndexResource(**i) for i in tool["azure_ai_search"]["indexes"]]
            return AzureAISearchTool(azure_ai_search=AzureAISearchToolResource(indexes=indexes))
        if tool["type"] == "web_search":
            args = {k: v for k, v in tool.items() if k not in ("type", "filters")}
            if tool.get("filters"):
                args["filters"] = WebSearchToolFilters(**tool["filters"])
            return WebSearchTool(**args)
        return MCPTool(**{k: v for k, v in tool.items() if k != "type"})

    with AIProjectClient(endpoint=out["FOUNDRYIQ_DEMO_PROJECT_ENDPOINT"], credential=credential) as project:
        for path in sorted((ROOT / "agents").glob("*.yaml")):
            missing = set()
            spec = substitute(yaml.safe_load(path.read_text(encoding="utf-8")), values, missing)
            if missing:
                raise ValueError(f"{path.name}: valori mancanti {', '.join(sorted(missing))}")
            definition = build_definition(spec)
            digest = hashlib.sha256(json.dumps({"definition": definition, "description": spec.get("description", "")},
                                               sort_keys=True).encode("utf-8")).hexdigest()
            try:
                latest = (project.agents.get(spec["name"]).versions or {}).get("latest")
                current = (latest or {}).get("metadata", {}).get("definition_sha256")
            except ResourceNotFoundError:
                current = None
            if current == digest:
                print(f"  invariato: {spec['name']}")
            else:
                extra = {"temperature": definition["temperature"]} if "temperature" in definition else {}
                version = project.agents.create_version(
                    agent_name=spec["name"], description=spec.get("description"),
                    definition=PromptAgentDefinition(model=definition["model"], instructions=definition["instructions"],
                                                     tools=[sdk_tool(t) for t in definition["tools"]], **extra),
                    metadata={"managed_by": "notebook", "definition_sha256": digest})
                print(f"  ok: {spec['name']} versione {version.version}")
            created.append({"oggetto": spec["name"], "tipo": f"agente ({len(definition['tools'])} tool)", "dove": "progetto Foundry"})
else:
    print("RUN_AGENTS=False: agenti non aggiornati.")

In [ ]:
import pandas as pd
from IPython.display import display

mcp_key = None
display(pd.DataFrame(created))
print(f"Chat demo: {out['CHAT_APP_URL']}")
print("Prossimo passo: notebook in notebooks/demo/ (multi-fonte, ACL, agenti).")